# IDS Practice with Suricata (remote execution in Jupyter)

In this practice you will learn to **install, configure and run Suricata** as an IDS engine for:
- **Live capture** (traffic from a network interface).
- **Offline analysis** of **PCAP** files.
- **Log inspection** (`fast.log` and `eve.json`) to interpret alerts and events.


<a href="https://colab.research.google.com/github/UPM-RSTI/RTVE/blob/main/SURI_IDS_MASTER_EN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# Open in Google Colab
> **Important:** run the cells **in order**.


## Introduction: Suricata and IDS

**Suricata** is a high-performance, open-source IDS/IPS engine. It provides:
- Real-time deep packet inspection (DPI).
- Signature-based (rule-based) detection and alert generation.
- Structured event logging (e.g. in `eve.json`).

An **IDS (Intrusion Detection System)** analyzes traffic or events to detect malicious or unauthorized activity. In this practice we focus on **detection and analysis**, interpreting the alerts Suricata produces.


## Requirements and recommendations

**Recommended environment**
- Linux (Ubuntu/Debian) with `sudo` privileges.
- Internet connection to install packages and download PCAPs.

**Before you start**
- If you're in a managed environment (e.g., a remote server or a VM), make sure you have:
  - `sudo` access.
  - Enough free space (a few hundred MB for rules and PCAPs).
  - Permission to capture traffic (only needed for live-capture mode).

**Path conventions used in this practice**
- Main configuration: `/etc/suricata/suricata.yaml`
- Updated rules (typical after `suricata-update`): `/var/lib/suricata/rules/suricata.rules`
- Typical logs: `/var/log/suricata/` (though this can vary if `-l` is used when running Suricata)


# IDS Practice – Suricata

Follow the steps in order. When you see a block with commands, run it and check the output before continuing.

In [ ]:
!sudo apt-get update
!sudo apt-get install -y suricata

## 1. Live traffic capture (optional)

In this section you will run Suricata listening on a **network interface** and generating logs in real time.

1) List the available interfaces and pick the one that matches your connection (e.g. `eth0`, `enp0s3`, `wlan0`).  
2) Run Suricata in capture mode on that interface.

> If your environment **does not allow live capture** (e.g., container or permission restrictions), skip to the **PCAP Analysis** section.


In [ ]:
# Run this command to list your network interfaces and find the name of the interface you want to monitor (e.g., eth0, enp0s3, wlan0).
!ip a

Once you have the interface name (e.g., `eth0`), start Suricata in capture mode.

- This command usually runs in the foreground.
- To stop it, use `Ctrl+C`.

> **Tip:** if you want to quickly check whether alerts are being generated, inspect the logs afterward (`/var/log/suricata/fast.log` and `/var/log/suricata/eve.json`).


In [ ]:
# Replace '<interface_name>' with your actual interface (e.g., eth0, enp0s3).
# This command requires superuser privileges.
# !sudo suricata -i <interface_name> -c /etc/suricata/suricata.yaml -s /etc/suricata/rules/suricata.rules

# Example (uncomment and adapt if you know your interface):
#!sudo suricata -i eth0 -c /etc/suricata/suricata.yaml -s /etc/suricata/rules/suricata.rules

## 2. Reviewing the Suricata configuration (`suricata.yaml`)

Before analyzing traffic, we'll review the configuration file to understand:
- Where Suricata looks for **rules**.
- How the network variables (`HOME_NET`, `EXTERNAL_NET`) are defined.
- Which **log outputs** are enabled (especially `fast.log` and `eve.json`).


In [ ]:
print('Contents of /etc/suricata/suricata.yaml:')
!sudo cat /etc/suricata/suricata.yaml

## 3. Key points in `suricata.yaml` (quick guide)

In this practice we'll focus especially on:
- **Rule path** (`default-rule-path` and `rule-files`)
- **Network variables** (`HOME_NET`, `EXTERNAL_NET`)
- **Port groups** (`port-groups`)
- **Log outputs** (`outputs`: `eve.json`, `fast.log`)

Below is an extended explanation to help you interpret the configuration:

### Guide to Important Attributes in `suricata.yaml`

The `suricata.yaml` file is the heart of Suricata's configuration. Below are some of the key attributes, with emphasis on the rule file, IP addresses, ports, `HOME_NET` and `EXTERNAL_NET`.

### 1. `default-rule-path`

*   **Location:** Usually found under the `rules:` section. Sets the base path where Suricata will look for rule files.
*   **Example:** `default-rule-path: /etc/suricata/rules`
*   **Importance:** Defines the main directory for your rules. `suricata-update` usually places the compiled rules in `/var/lib/suricata/rules/`.

### 2. `rule-files`

*   **Location:** Also under the `rules:` section, specifies which individual rule files Suricata should load.
*   **Example:**
    ```yaml
    rule-files:
      - suricata.rules
      # - app-layer-events.rules
    ```
*   **Importance:** This is where you tell Suricata which rule collections to use. As shown earlier, if `suricata-update` places the rules in `/var/lib/suricata/rules/suricata.rules`, it's crucial that this section points to that location, or that the `suricata.rules` file in `/etc/suricata/rules/` contains a correct reference.

### 3. Network Configuration: `HOME_NET` and `EXTERNAL_NET`

These are two of the most critical parameters in Suricata, since they define what counts as your internal (protected) network and what is external to it. Suricata rules often use these variables to distinguish between internal and external traffic, which is essential for accurate detection.

*   **`HOME_NET`**
    *   **Location:** Under the `vars:` section. Defines the IP address range of your internal network, or the network you want to protect.
    *   **Example:** `HOME_NET: "[192.168.0.0/16,10.0.0.0/8,172.16.0.0/12]"`
    *   **Importance:** All IP addresses within this range are considered 'local' or 'internal'. Suricata rules use this to distinguish attacks coming from outside (external to internal) from internal ones (internal to internal, or internal to external).

*   **`EXTERNAL_NET`**
    *   **Location:** Under the `vars:` section. Defines all IP addresses that are not part of `HOME_NET` (generally, the Internet or any other network outside your control).
    *   **Example:** `EXTERNAL_NET: "!$HOME_NET"`
    *   **Importance:** `!$HOME_NET` is the most common and safest way to configure it, meaning "everything that is NOT in HOME_NET". This ensures that all non-internal traffic is treated as external, simplifying rule writing and maintenance.

### 4. `port-groups`

*   **Location:** Under the `vars:` section. Lets you define groups of ports commonly used by different services.
*   **Example:**
    ```yaml
    vars:
      # port groups
      HTTP_PORTS: "80,8080,8000,8008,8888,8443,443"
      FILE_DATA_PORTS: "$HTTP_PORTS,110,143,443,465,993,995,20,21,25,3306,5432,5900,8000,8080,8443,8888"
    ```
*   **Importance:** These variables are used in rules so you don't have to list every port in each rule, making rules more readable and easier to maintain. For example, a rule for HTTP traffic can simply use `$HTTP_PORTS` instead of `80,8080,...`.

### 5. `outputs` (Log Outputs)

*   **Location:** A top-level section in the `suricata.yaml` file.
*   **Example (relevant):**
    ```yaml
    outputs:
      - eve-log:
          enabled: yes
          filetype: regular #regular|syslog|unix_dgram|unix_stream
          filename: eve.json
          # Appending to an existing file can be done by setting 'append' to 'yes'.
          # append: no
          # If you want to dump alerts to a separate file, you can do this:
          # - alert:
          #     enabled: yes
          #     # filetype: regular # regular|syslog|unix_dgram|unix_stream
          #     # filename: alerts.json
          #     # append: yes
      - fast.log:
          enabled: yes
          filetype: regular
          # By default, messages are appended to fast.log.
          # Change to 'no' to overwrite at startup.
          # append: yes
          filename: fast.log
          # The fast log format: tag, timestamp, alert info
          # If you want to specify a specific format for the fast log, you can use:
          # format: "%T.%{usec} %F %{%H:%M:%S} %{%Y-%m-%d} %i %s %p %d %H %M %{alert.signature} %{alert.gid} %{alert.rev} %{alert.signature_id} %{event_type} %{src_ip} %{dest_ip} %{src_port} %{dest_port} %{proto}"
    ```
*   **Importance:** This section configures where and how Suricata stores its logs. We've set up `eve.json` for detailed events (including alerts) and `fast.log` for concise alerts. The path where these log files are written can be overridden with the `-l` option when running Suricata (as we did with `/tmp/suricata_logs`).

Understanding and adjusting these settings is essential for effective network monitoring with Suricata.

## 4. PCAP file analysis

Now we'll run Suricata in **offline** mode to analyze an already-captured PCAP.

- This mode is ideal for remote practices, since it doesn't require live traffic capture.
- After the analysis, we'll review the logs to identify relevant alerts and events.


In [ ]:
# Replace '<path_to_pcap_file>' with the path to your PCAP file.
# This command requires superuser privileges.
# !sudo suricata -r <path_to_pcap_file> -c /etc/suricata/suricata.yaml -s /etc/suricata/rules/suricata.rules

# Example (uncomment and adapt if you have a PCAP file):
# !sudo suricata -r /home/user/my_capture.pcap -c /etc/suricata/suricata.yaml -s /etc/suricata/rules/suricata.rules

## 5. Viewing Suricata logs

Suricata typically generates at least two useful files:

- `fast.log`: a **summarized** list of alerts (quick to inspect).
- `eve.json`: events in **JSON** format (more detailed; useful for searching, filtering and parsing).

In the following cells we'll display both.
> If the files don't show up in the current directory, also check `/var/log/suricata/`.


In [ ]:
# Show the contents of the fast.log log
!sudo cat /var/log/suricata/fast.log

In [ ]:
# Show the last lines of the eve.json log in real time (requires Suricata to be running and generating logs)
!sudo cat /var/log/suricata/eve.json

## 6. Downloading and extracting a sample PCAP

We'll download a password-protected PCAP archive (a real case published for analysis).
- Download the `.zip`
- Extract it using the given password
- Verify that the `.pcap` was extracted correctly


## 7. Updating the rules

Before analyzing traffic, we update the rules to maximize detection capability.

- `suricata-update` downloads and compiles rules into a path that is usually:  
  `/var/lib/suricata/rules/suricata.rules`

> If Suricata can't find rules or shows path warnings, check the rule configuration section in `suricata.yaml`.


In [ ]:
!sudo suricata-update

In [ ]:
# Download the .pcap.zip file
!wget https://www.malware-traffic-analysis.net/2024/11/24/2024-11-24-webserver-scans-and-probes.pcap.zip

# Extract the file to get the .pcap using the password
!unzip -P infected_20241124 2024-11-24-webserver-scans-and-probes.pcap.zip

# List the files to confirm the .pcap was extracted
!ls -l

In [ ]:
import subprocess

# Back up the original suricata.yaml file before modifying it
!sudo cp /etc/suricata/suricata.yaml /etc/suricata/suricata.yaml.bak

# Update the default rule path with the new one.
# This assumes the default rule is specified as '- suricata.rules'
# and that it lives inside the rules directory specified by 'default-rule-path',
# which is usually '/etc/suricata/rules/'.
# We'll replace the line pointing to the old rules file with the new one.

# First, identify the line to replace. It's normally a line like '- suricata.rules'
# under the 'rule-files:' section.
# The suricata-update tool creates a single 'suricata.rules' file in /var/lib/suricata/rules/
# So we need to comment out any existing rule file entry and add the new one.

# Read the original file contents
with open('/etc/suricata/suricata.yaml', 'r') as f:
    lines = f.readlines()

new_lines = []
rule_files_section = False
inserted_new_rule = False

for line in lines:
    if 'rule-files:' in line:
        new_lines.append(line) # Keep the 'rule-files:' line
        rule_files_section = True
        # Add the updated rule path right after the 'rule-files:' declaration
        indent = len(line) - len(line.lstrip()) # Preserve indentation
        new_lines.append(f'{' ' * indent}- /var/lib/suricata/rules/suricata.rules\n')
        inserted_new_rule = True
    elif rule_files_section and line.strip().startswith('- '):
        # Comment out old rule file entries if they're in the default path
        if '/etc/suricata/rules/' in line or 'suricata.rules' in line:
            new_lines.append(f'# {line}')
        else:
            new_lines.append(line) # Keep other rule files that may still be valid
    elif rule_files_section and not line.strip(): # End of the rule-files section (blank line)
        rule_files_section = False
        new_lines.append(line)
    else:
        new_lines.append(line)

# If the rule-files section wasn't found but we still need to add the rule
if not inserted_new_rule:
    # This case could be more complex if the 'rule-files' section doesn't exist at all, or is structured differently.
    # For this specific solution, we assume 'rule-files' exists and we're modifying it.
    # If it didn't exist, we would need to insert it strategically.
    # To keep things simple, if it's not found and not inserted, we assume it's already handled or not relevant.
    # In a real scenario, a more robust analysis might be needed.
    pass # No action needed if the rule was inserted correctly above, or if there's no rule-files section to begin with

# Write the modified content back to suricata.yaml
with open('/etc/suricata/suricata.yaml', 'w') as f:
    f.writelines(new_lines)

print("suricata.yaml updated to use rules from /var/lib/suricata/rules/suricata.rules")

In [ ]:
import os

pcap_file_path = '/content/2024-11-24-webserver-scans-and-probes.pcap'
config_file_path = '/etc/suricata/suricata.yaml'
rules_file_path = '/var/lib/suricata/rules/suricata.rules' # This path is now also set in suricata.yaml

# Make sure the PCAP file exists before trying to analyze it
if not os.path.exists(pcap_file_path):
    print(f"Error: PCAP file not found at {pcap_file_path}")
else:
    print(f"Re-analyzing the PCAP file: {pcap_file_path} with the updated configuration...")
    !sudo suricata -r {pcap_file_path} -c {config_file_path} --runmode autofp

# Note: We could now omit the -s argument since the rules path is set in the config file,
# but keeping it explicit here is also harmless, since it overrides the config if different.
# However, to truly test the configuration file change, we can remove -s.
# Let's remove -s to confirm the configuration file change took effect for rule loading.
# Re-running without -s to rely solely on the updated suricata.yaml for the rules path.
print("Re-analyzing the PCAP file again, relying on the updated suricata.yaml for the rules path...")
!sudo suricata -r {pcap_file_path} -c {config_file_path} --runmode autofp

In [ ]:
print("Contents of fast.log after the WannaCry analysis:")
!sudo cat fast.log

In [ ]:
print("Contents of eve.json after the WannaCry analysis:")
!sudo cat eve.json

## 8. Case study: WannaCry / EternalBlue (historical PCAP)

In this section we'll analyze a historical PCAP associated with the **EternalBlue** exploit (MS17-010) used in campaigns such as WannaCry.

Objective:
- Run Suricata against the PCAP.
- Inspect `fast.log` and `eve.json` for relevant alerts.


In [ ]:
pcap_zip_url = 'https://www.malware-traffic-analysis.net/2017/05/18/2017-05-18-WannaCry-ransomware-using-EnternalBlue-exploit.pcap.zip'
pcap_password = 'infected_20170518'

# Download the .pcap.zip file
print(f"Downloading file: {pcap_zip_url}")
!wget {pcap_zip_url}

# Extract the file to get the .pcap using the password
pcap_zip_filename = pcap_zip_url.split('/')[-1]
print(f"Extracting file: {pcap_zip_filename} with password...")
!unzip -P {pcap_password} {pcap_zip_filename}

# List the files to confirm the .pcap was extracted
print("Listing files in the current directory:")
!ls -l

In [ ]:
import os

pcap_file_path = '/content/2017-05-18-WannaCry-ransomware-using-EnternalBlue-exploit.pcap'
config_file_path = '/etc/suricata/suricata.yaml'
rules_file_path = '/var/lib/suricata/rules/suricata.rules' # This path is now also set in suricata.yaml

# Make sure the PCAP file exists before trying to analyze it
if not os.path.exists(pcap_file_path):
    print(f"Error: PCAP file not found at {pcap_file_path}")
else:
    print(f"Re-analyzing the PCAP file: {pcap_file_path} with the updated configuration...")
    !sudo suricata -r {pcap_file_path} -c {config_file_path} --runmode autofp

# Note: We could now omit the -s argument since the rules path is set in the config file,
# but keeping it explicit here is also harmless, since it overrides the config if different.
# However, to truly test the configuration file change, we can remove -s. Let's remove -s
# to confirm the configuration file change took effect for rule loading.
print("Re-analyzing the PCAP file again, relying on the updated suricata.yaml for the rules path...")
!sudo suricata -r {pcap_file_path} -c {config_file_path} --runmode autofp

In [ ]:
print("Contents of fast.log after the WannaCry analysis:")
!sudo cat fast.log

In [ ]:
print("Contents of eve.json after the WannaCry analysis:")
!sudo cat eve.json